# pa2-generate — Task 6: evaluation generations + blinded pairwise files
Settings: GPU T4 x2, Internet on. **Add Input → Notebook Output**: pa2-sft, pa2-dpo-hh, pa2-dpo-uf, pa2-lora-persona.
Run the sanity cell interactively first if you like, then **Save & Run All (Commit)**. ~30–60 min.
Decoding (identical for every model in a comparison): greedy, repetition_penalty 1.1, max_new_tokens 512.

In [ ]:
# ---- Setup: identical first cell in every notebook (a commit starts on a fresh machine) ----
GH_USER, REPO = "mohdddd", "small-model-alignment-starter"
%cd /kaggle/working
!rm -rf repo && git clone -q https://github.com/{GH_USER}/{REPO}.git repo
%cd /kaggle/working/repo
!git log --oneline -1
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps
import os, sys, json, glob, shutil, subprocess
sys.path.insert(0, "/kaggle/working/repo/src")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs("outputs", exist_ok=True)

In [ ]:
!python -m pip install -q "peft>=0.13"

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch, transformers, datasets
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| transformers", transformers.__version__,
      "| datasets", datasets.__version__, "| GPUs:", torch.cuda.device_count())

In [ ]:
from pa2.utils import find_one
d = os.path.dirname
P = {
    "sft": d(find_one("sft/best/model.safetensors")),
    "dpo_hh": d(find_one("dpo_hh/best/model.safetensors")),
    "dpo_uf": d(find_one("dpo_uf/best/model.safetensors")),
    "lora": d(find_one("lora_persona/adapter/adapter_config.json")),
}
ALPACA = "outputs/data/alpaca_eval_300_seed42.json"
HH, PERSONA = "eval_sets/hh_rlhf_eval_300_seed42.json", "eval_sets/persona_style_eval_24.json"
SMOL = "HuggingFaceTB/SmolLM2-360M-Instruct"
os.makedirs("outputs/data", exist_ok=True); os.makedirs("outputs/generations", exist_ok=True)
shutil.copy(find_one("data/alpaca_eval_300_seed42.json"), ALPACA)
for f in ("alpaca_base.json", "hh_base.json"):
    shutil.copy(find_one("generations/" + f), "outputs/generations/" + f)
print(json.dumps(P, indent=1))

In [ ]:
def gen(model, eval_set, out, name, device, extra=""):
    return (f"python scripts/generate.py --model {model} --eval_set {eval_set} --out outputs/generations/{out}.json "
            f"--generator {name} --device {device} {extra}")

G0 = [gen(P["sft"], ALPACA, "alpaca_sft", "qwen2.5-0.5b-sft", "cuda:0"),
      gen(P["dpo_uf"], ALPACA, "alpaca_dpo_uf", "qwen2.5-0.5b-dpo-uf", "cuda:0"),
      gen(SMOL, PERSONA, "persona_smol_base", "smollm2-360m-instruct", "cuda:0", "--template chat")]
G1 = [gen(P["sft"], HH, "hh_sft", "qwen2.5-0.5b-sft", "cuda:1"),
      gen(P["dpo_hh"], HH, "hh_dpo_hh", "qwen2.5-0.5b-dpo-hh", "cuda:1"),
      gen(SMOL, PERSONA, "persona_lora", "smollm2-360m-lora-dpo", "cuda:1", f"--template chat --adapter {P['lora']}")]
RUN_CROSS = False  # optional alignment-tax check: HH-DPO on AlpacaEval, UF-DPO on the HH set (+~15 min)
if RUN_CROSS:
    G0.append(gen(P["dpo_hh"], ALPACA, "alpaca_dpo_hh", "qwen2.5-0.5b-dpo-hh", "cuda:0"))
    G1.append(gen(P["dpo_uf"], HH, "hh_dpo_uf", "qwen2.5-0.5b-dpo-uf", "cuda:1"))

Sanity check (2–5 prompts per model) — read the outputs before the full run.

In [ ]:
for c in [G0[0], G0[1], G1[1], G1[2]]:
    !{c} --limit 3 --max_new_tokens 256 2>&1 | grep -v Warning | tail -12

Full generation: two queues in parallel (cuda:0 and cuda:1). Each file is written as soon as its model finishes; existing files are skipped, so a rerun resumes.

In [ ]:
procs = []
for i, chain in enumerate([G0, G1]):
    log = open(f"outputs/generation_gpu{i}.log", "w")
    procs.append((subprocess.Popen(" && ".join(chain), shell=True, stdout=log, stderr=subprocess.STDOUT), log))
for p, log in procs:
    print("exit code", p.wait()); log.close()
for i in range(2):
    print(f"--- gpu{i} log tail ---"); print(open(f"outputs/generation_gpu{i}.log").read()[-1500:])

In [ ]:
# Blinded pairwise files (random A/B order, seed 42; identities only in hidden meta)
!python scripts/build_pairs.py --candidate outputs/generations/alpaca_dpo_uf.json --baseline outputs/generations/alpaca_sft.json --out outputs/pairs/uf_dpo_vs_sft_alpaca.jsonl --mode general
!python scripts/build_pairs.py --candidate outputs/generations/hh_dpo_hh.json --baseline outputs/generations/hh_sft.json --out outputs/pairs/hh_dpo_vs_sft_hh.jsonl --mode safety
!python scripts/build_pairs.py --candidate outputs/generations/persona_lora.json --baseline outputs/generations/persona_smol_base.json --out outputs/pairs/lora_vs_base_persona.jsonl --mode persona

In [ ]:
# Length / truncation stats per generation file (for the verbosity discussion in the report)
import statistics
for f in sorted(glob.glob("outputs/generations/*.json")):
    if f.endswith(".settings.json") or ".sanity" in f: continue
    rows = json.load(open(f))
    words = [len(r["output"].split()) for r in rows]
    print(f"{os.path.basename(f):28s} n={len(rows):3d}  mean_words={statistics.mean(words):6.1f}  "
          f"median={statistics.median(words):5.0f}  hit_max_tokens={sum(r['hit_max_new_tokens'] for r in rows):3d}  "
          f"empty={sum(not r['output'] for r in rows)}")

In [ ]:
!cd outputs && zip -qr /kaggle/working/generation_outputs.zip generations pairs data *.log
!du -sh /kaggle/working/repo/outputs/* 2>/dev/null; df -h /kaggle/working | tail -1